In [5]:
from fuzzycar import MAXSONAR
from pynq.overlays.base import BaseOverlay
from pynq.lib import MicroblazeLibrary
import time
from IPython.display import clear_output
from collections import deque

# Initialize
base = BaseOverlay('base.bit')
arduino_lib = MicroblazeLibrary(base.iop_arduino, ['uart'])
sonar = MAXSONAR(base, arduino_lib)

# Keep track of recent readings
history = deque(maxlen=50)  # Last 50 readings
start_time = time.time()

try:
    print("Starting MAXSONAR readings...")
    print("(Press Ctrl+C to exit)")
    
    while True:
        # Read distance
        distance = sonar.read_distance()
        history.append(distance)
        
        # Clear previous output
        clear_output(wait=True)
        
        # Display reading and statistics
        print("MAXSONAR Real-time Analysis")
        print("=" * 40)
        
        # Current reading
        print("\nCurrent Reading:")
        print("-" * 15)
        if distance == -1:
            print("Error reading sensor")
        elif distance == 255:
            print("No object detected (maximum range)")
        elif distance == 6:
            print("Object at minimum range (6 inches)")
        else:
            print(f"Distance: {distance} inches ({distance * 2.54:.1f} cm)")
        
        # Statistics from history
        if len(history) > 1:
            valid_readings = [x for x in history if x > 6 and x < 255]
            if valid_readings:
                print("\nStatistics:")
                print("-" * 15)
                print(f"Average: {sum(valid_readings)/len(valid_readings):.1f} inches")
                print(f"Minimum: {min(valid_readings)} inches")
                print(f"Maximum: {max(valid_readings)} inches")
                
                # Detect if object is moving
                if len(valid_readings) > 2:
                    delta = valid_readings[-1] - valid_readings[-2]
                    if abs(delta) > 2:  # More than 2 inches change
                        if delta > 0:
                            print("Object moving away ↑")
                        else:
                            print("Object moving closer ↓")
                    else:
                        print("Object stationary ≈")
        
        # Visualization
        print("\nDistance Map (each # = 2 inches):")
        print("0" + "-" * 49 + "100 inches")
        if 6 < distance < 255:
            bar = "#" * (distance // 2)
            print(f"{bar:<50} {distance}")
        
        # Running time
        elapsed = time.time() - start_time
        print(f"\nRunning Time: {elapsed:.1f} seconds")
        print(f"Sample Count: {len(history)}")
        print(f"Sample Rate: {len(history)/elapsed:.1f} Hz")
        
        time.sleep(0.05)  # 10Hz update rate

except KeyboardInterrupt:
    print("\nExiting...")
finally:
    sonar.close()
    print("Sensor disconnected")

MAXSONAR Real-time Analysis

Current Reading:
---------------
Distance: 17 inches (43.2 cm)

Statistics:
---------------
Average: 78.5 inches
Minimum: 16 inches
Maximum: 157 inches
Object stationary ≈

Distance Map (each # = 2 inches):
0-------------------------------------------------100 inches
########                                           17

Running Time: 32.0 seconds
Sample Count: 50
Sample Rate: 1.6 Hz

Exiting...
Sensor disconnected
